In [ ]:
import sys
import os

data_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if data_root not in sys.path:
    sys.path.insert(0, data_root)


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from data.utils.feature_extraction import get_sld, get_tld
from data.utils.fold_splitting import analyze_folds
from data.utils.general import gini_coefficient, remove_protocol_and_www

In [ ]:
class StatisticalResult:
    values: list[float]
    count: int
    mean: float
    median: float
    std: float
    min: float
    max: float
    gini: float

    def __init__(self, values: list[float]):
        self.values = np.array(values)
        self.count = len(self.values)
        
        # Handle empty arrays
        if self.count == 0:
            self.mean = 0.0
            self.median = 0.0
            self.std = 0.0
            self.min = 0.0
            self.max = 0.0
            self.gini = 0.0
        else:
            self.mean = np.mean(self.values)
            self.median = np.median(self.values)
            self.std = np.std(self.values)
            self.min = np.min(self.values)
            self.max = np.max(self.values)
            self.gini = gini_coefficient(self.values)

class TLDResult:
    tld_counts: dict[str, int]
    tld_percentages: dict[str, float]
    stats: StatisticalResult

    def __init__(self, tld_counts: dict[str, int]):
        self.tld_counts = tld_counts
        self.tld_percentages = {k: v / sum(self.tld_counts.values()) for k, v in self.tld_counts.items()}
        self.stats = StatisticalResult(list(self.tld_percentages.values()))

    def get_top_n_tld_percentages(self, n: int) -> list[tuple[str, float]]:
        return sorted(self.tld_percentages.items(), key=lambda x: x[1], reverse=True)[:n]

class DatasetExplorationResult:
    dataset_name: str
    df: pd.DataFrame
    type: str
    benign_count: int
    malicious_count: int
    total_count: int
    benign_percentage: float
    malicious_percentage: float

    duplicates_percentage: float

    length: StatisticalResult
    length_benign: StatisticalResult
    length_malicious: StatisticalResult
    
    # Add stripped URL length statistics
    length_stripped: StatisticalResult
    length_stripped_benign: StatisticalResult  
    length_stripped_malicious: StatisticalResult

    sld_result: StatisticalResult

    tld_total: TLDResult
    tld_benign: TLDResult
    tld_malicious: TLDResult

    fold_results: list[pd.DataFrame]

    sld_pct_full_benign: float
    sld_pct_full_malicious: float
    
    fold_sld_leakage_results: list[dict]
    avg_fold_sld_leakage: float

    def calculate_leakage(self, fold_column: str):
        """Calculate SLD leakage across folds using specified fold column"""
        self.fold_sld_leakage_results = []
        unique_folds = sorted(self.df[fold_column].unique())
        
        for eval_fold in unique_folds:
            # Split data into training and evaluation sets
            train_df = self.df[self.df[fold_column] != eval_fold]
            eval_df = self.df[self.df[fold_column] == eval_fold]
            
            if len(train_df) == 0 or len(eval_df) == 0:
                self.fold_sld_leakage_results.append({
                    'fold': eval_fold,
                    'eval_urls_total': len(eval_df),
                    'eval_urls_with_pure_slds': 0,
                    'leakage_percentage': 0.0
                })
                continue
            
            # Find SLDs in training set that are either fully benign or fully malicious
            train_sld_label_counts = train_df.groupby('sld')['label'].value_counts().unstack(fill_value=0)
            
            # Ensure both columns exist
            if 'benign' not in train_sld_label_counts.columns:
                train_sld_label_counts['benign'] = 0
            if 'malicious' not in train_sld_label_counts.columns:
                train_sld_label_counts['malicious'] = 0
            
            # Identify pure SLDs in training set
            train_fully_benign_slds = train_sld_label_counts[
                (train_sld_label_counts['benign'] > 0) & (train_sld_label_counts['malicious'] == 0)
            ]
            train_fully_malicious_slds = train_sld_label_counts[
                (train_sld_label_counts['malicious'] > 0) & (train_sld_label_counts['benign'] == 0)
            ]
            
            # Combine both sets of pure SLD names
            pure_sld_names = set(train_fully_benign_slds.index).union(set(train_fully_malicious_slds.index))
            
            # Count how many evaluation URLs have SLDs from the pure training SLDs
            eval_urls_with_pure_slds = eval_df[eval_df['sld'].isin(pure_sld_names)].shape[0]
            total_eval_urls = len(eval_df)
            
            leakage_percentage = (eval_urls_with_pure_slds / total_eval_urls * 100) if total_eval_urls > 0 else 0.0
            
            self.fold_sld_leakage_results.append({
                'fold': eval_fold,
                'eval_urls_total': total_eval_urls,
                'eval_urls_with_pure_slds': eval_urls_with_pure_slds,
                'leakage_percentage': leakage_percentage
            })
            
            print(f"Fold {eval_fold}: {eval_urls_with_pure_slds}/{total_eval_urls} eval URLs ({leakage_percentage:.2f}%) have pure training SLDs")
        
        # Calculate average leakage across all folds
        avg_leakage = np.mean([result['leakage_percentage'] for result in self.fold_sld_leakage_results])
        self.avg_fold_sld_leakage = avg_leakage
        print(f"Average SLD leakage across folds: {avg_leakage:.2f}%")

    def __init__(self, df: pd.DataFrame, type: str, name: str):
        self.df = df
        self.type = type
        self.name = name
        print(f"Processing dataset: {name}, getting sld")
        df["sld"] = df["url"].map(get_sld)
        print("Getting tld")
        df["tld"] = df["url"].map(get_tld)
        print("Getting stripped url lengths")
        df["url_stripped"] = df["url"].map(remove_protocol_and_www)
        
        self.benign_count = df[df["label"] == "benign"].shape[0]
        self.malicious_count = df[df["label"] == "malicious"].shape[0]
        self.total_count = self.benign_count + self.malicious_count
        self.benign_percentage = self.benign_count / self.total_count
        self.malicious_percentage = self.malicious_count / self.total_count
        self.duplicates_percentage = df[df.duplicated(subset="url")].shape[0] / self.total_count
        
        # Original URL length statistics
        self.length = StatisticalResult(df["url"].str.len().tolist())
        self.length_benign = StatisticalResult(df[df["label"] == "benign"]["url"].str.len().tolist())
        self.length_malicious = StatisticalResult(df[df["label"] == "malicious"]["url"].str.len().tolist())
        
        # Stripped URL length statistics (without protocol and www)
        self.length_stripped = StatisticalResult(df["url_stripped"].str.len().tolist())
        self.length_stripped_benign = StatisticalResult(df[df["label"] == "benign"]["url_stripped"].str.len().tolist())
        self.length_stripped_malicious = StatisticalResult(df[df["label"] == "malicious"]["url_stripped"].str.len().tolist())
        
        self.sld_result = StatisticalResult(df["sld"].value_counts().tolist())
        
        self.tld_total = TLDResult(df["tld"].value_counts().to_dict())
        self.tld_benign = TLDResult(df[df["label"] == "benign"]["tld"].value_counts().to_dict())
        self.tld_malicious = TLDResult(df[df["label"] == "malicious"]["tld"].value_counts().to_dict())

        # Handle fold analysis - fix the indentation issue
        if "fold" in df.columns:
            self.fold_results = analyze_folds(df)
        else:
            self.fold_results = None
            # Create random folds only if no fold column exists
        df["random_fold"] = np.random.randint(0, 5, len(df))
            
        # Calculate SLD purity statistics
        sld_label_counts = df.groupby('sld')['label'].value_counts().unstack(fill_value=0)

        # Ensure both 'benign' and 'malicious' columns exist
        if 'benign' not in sld_label_counts.columns:
            sld_label_counts['benign'] = 0
        if 'malicious' not in sld_label_counts.columns:
            sld_label_counts['malicious'] = 0

        fully_benign_slds = sld_label_counts[(sld_label_counts['benign'] > 0) & (sld_label_counts['malicious'] == 0)]
        fully_malicious_slds = sld_label_counts[(sld_label_counts['malicious'] > 0) & (sld_label_counts['benign'] == 0)]
        total_slds = len(sld_label_counts)

        self.sld_pct_full_benign = len(fully_benign_slds) / total_slds * 100 if total_slds > 0 else 0.0
        self.sld_pct_full_malicious = len(fully_malicious_slds) / total_slds * 100 if total_slds > 0 else 0.0
        
        # Calculate leakage using appropriate fold column
        self.calculate_leakage("random_fold")

In [ ]:
dataset_results: DatasetExplorationResult = []

datasets_path = "public_dataset"

all_dfs = []

for subcsv in Path(datasets_path, "full_malicious_datasets").glob("*/dataset.csv"):
    if not subcsv.exists():
        continue
    print(f"Reading malicious: {subcsv}")
    name = subcsv.parent.name
    df = pd.read_csv(subcsv, usecols=["url"])
    df["label"] = "malicious"
    dataset_results.append(DatasetExplorationResult(df, "full_mal", name))

for subcsv in Path(datasets_path, "good_datasets").glob("*/dataset.csv"):
    if not subcsv.exists():
        continue
    print(f"Reading mixed: {subcsv}")
    name = subcsv.parent.name
    cols = pd.read_csv(subcsv, nrows=0).columns.tolist()
    usecols = [c for c in ["url", "label"] if c in cols]
    df = pd.read_csv(subcsv, usecols=usecols)
    dataset_results.append(DatasetExplorationResult(df, "mixed", name))

benign_csv = Path(datasets_path, "benign_commoncrawl/dataset.csv")
if benign_csv.exists():
    print(f"Reading benign: {benign_csv}")
    df = pd.read_csv(benign_csv, usecols=["url"])
    df["label"] = "benign"
    dataset_results.append(DatasetExplorationResult(df, "full_ben", "benign_commoncrawl"))

merged_dataset = Path(datasets_path, "dataset.csv")
merged_df = pd.read_csv(merged_dataset)
dataset_results.append(DatasetExplorationResult(merged_df, "mixed", "final_merged"))

print(dataset_results)


In [ ]:
for dr in dataset_results:
    dr.full_name = dr.name
    dr.name = dr.name[:30]

In [ ]:
dataset_results = [dataset_result if dataset_result.name != "final_merged" else None for dataset_result in dataset_results]
dataset_results.append(DatasetExplorationResult(merged_df, "mixed", "final_merged"))

In [ ]:
new_dataset_reuslts = []
for result in dataset_results:
    if result != None:
        new_dataset_reuslts.append(result)

dataset_results = new_dataset_reuslts

In [ ]:

overview_data = []
for result in dataset_results:
    overview_data.append({
        'Dataset': result.name,
        'Type': result.type,
        'Total URLs': f"{result.total_count:,}",
        'Benign': f"{result.benign_count:,} ({result.benign_percentage*100:.1f}%)",
        'Malicious': f"{result.malicious_count:,} ({result.malicious_percentage*100:.1f}%)",
        'Duplicates': f"{result.duplicates_percentage*100:.2f}%"
    })

overview_df = pd.DataFrame(overview_data)
display(overview_df)
print()

In [ ]:
# find it in array
final_dataset = [dataset for dataset in dataset_results if dataset.name == "final_merged"][0]
df = final_dataset.df
print(len(df[df["dataset_name"] == "phishcrawl"]))

In [ ]:
# URL Length Statistics
length_data = []
for result in dataset_results:
    length_data.append({
        'Dataset': result.name,
        'Original Mean±Std': f"{result.length.mean:.1f}±{result.length.std:.1f}",
        'Original Median': f"{result.length.median:.1f}",
        'Original Benign Mean±Std': f"{result.length_benign.mean:.1f}±{result.length_benign.std:.1f}",
        'Original Benign Median': f"{result.length_benign.median:.1f}",
        'Original Malicious Mean±Std': f"{result.length_malicious.mean:.1f}±{result.length_malicious.std:.1f}",
        'Original Malicious Median': f"{result.length_malicious.median:.1f}",
        'Stripped Mean±Std': f"{result.length_stripped.mean:.1f}±{result.length_stripped.std:.1f}",
        'Stripped Median': f"{result.length_stripped.median:.1f}",
        'Stripped Benign Mean±Std': f"{result.length_stripped_benign.mean:.1f}±{result.length_stripped_benign.std:.1f}",
        'Stripped Benign Median': f"{result.length_stripped_benign.median:.1f}",
        'Stripped Malicious Mean±Std': f"{result.length_stripped_malicious.mean:.1f}±{result.length_stripped_malicious.std:.1f}",
        'Stripped Malicious Median': f"{result.length_stripped_malicious.median:.1f}"
    })

print("URL Length Statistics")
display(pd.DataFrame(length_data).set_index('Dataset'))

In [ ]:
# SLD Analysis
sld_data = []
for result in dataset_results:
    sld_data.append({
        'Dataset': result.name,
        'Total URLs': f"{result.total_count:,}",
        'Unique SLDs': f"{result.sld_result.count:,}",
        'URLs per SLD Mean±Std': f"{result.sld_result.mean:.1f}±{result.sld_result.std:.1f}",
        'URLs per SLD Median': f"{result.sld_result.median:.1f}",
        'SLD Gini': f"{result.sld_result.gini:.3f}",
        'Pure Benign SLDs %': f"{result.sld_pct_full_benign:.1f}%",
        'Pure Malicious SLDs %': f"{result.sld_pct_full_malicious:.1f}%",
        'Avg Fold Leakage %': f"{result.avg_fold_sld_leakage:.1f}%"
    })

print("SLD Analysis")
display(pd.DataFrame(sld_data).set_index('Dataset'))

In [ ]:
# TLD Analysis
tld_data = []
n = 10
for result in dataset_results:
    # Get top 5 TLDs for each category
    top5_total = result.tld_total.get_top_n_tld_percentages(n)
    top5_benign = result.tld_benign.get_top_n_tld_percentages(n)
    top5_malicious = result.tld_malicious.get_top_n_tld_percentages(n)
    
    tld_data.append({
        'Dataset': result.name,
        'Total TLD Gini': f"{result.tld_total.stats.gini:.3f}",
        f'Top {n} Total': ', '.join([f"{tld}({pct:.1%})" for tld, pct in top5_total]),
        'Benign TLD Gini': f"{result.tld_benign.stats.gini:.3f}",
        f'Top {n} Benign': ', '.join([f"{tld}({pct:.1%})" for tld, pct in top5_benign]),
        'Malicious TLD Gini': f"{result.tld_malicious.stats.gini:.3f}",
        f'Top {n} Malicious': ', '.join([f"{tld}({pct:.1%})" for tld, pct in top5_malicious])
    })

print("TLD Analysis")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
tld_df = pd.DataFrame(tld_data).set_index('Dataset')
display(tld_df.style.set_properties(**{"white-space": "pre-wrap"}))

In [ ]:
# Leakage Analysis
leakage_data = []
for result in dataset_results:
    leakage_data.append({
        'Dataset': result.name,
        'Pure Benign SLDs %': f"{result.sld_pct_full_benign:.1f}%",
        'Pure Malicious SLDs %': f"{result.sld_pct_full_malicious:.1f}%",
        'Total Pure SLDs %': f"{result.sld_pct_full_benign + result.sld_pct_full_malicious:.1f}%",
        'Avg Fold Leakage %': f"{result.avg_fold_sld_leakage:.1f}%",
        'Min Fold Leakage %': f"{min([r['leakage_percentage'] for r in result.fold_sld_leakage_results]):.1f}%" if result.fold_sld_leakage_results else "N/A",
        'Max Fold Leakage %': f"{max([r['leakage_percentage'] for r in result.fold_sld_leakage_results]):.1f}%" if result.fold_sld_leakage_results else "N/A"
    })

print("Leakage Analysis")
display(pd.DataFrame(leakage_data).set_index('Dataset'))